# Table 3 (two columns)
This notebook outputs only IS_R2_head and OOS_R2_head for the monthly predictors.

## Sources / Replication provenance
- Goyal, A. & Welch, I. (2008). *A Comprehensive Look at The Empirical Performance of Equity Premium Prediction*.
- Adapted from the replication repo: https://github.com/juliusd01/ReplicationGoyalWelch2008
- I rewrote and simplified substantial parts for teaching and clarity, and validated outputs against Table 3 targets.

## Beginner notes (what to watch for)
- **Dates matter:** we convert `yyyymm` to a real date and set it as the index. All regressions use **lagged predictors**, so the first row becomes `NaN` and is dropped.
- **Lag choice:** most predictors use a 1‑month lag; `infl` uses a 2‑month lag (data availability).
- **Out‑of‑sample (OOS):** we use a rolling 20‑year window (240 months). Each month’s model is fit only on data available **up to that month**.
- **Expected sign rule:** if the estimated slope flips sign, we set it to zero (per the paper’s constraint).
- **Scale:** results are reported in **percent** (multiply by 100).
- **Reproducibility:** small changes to end dates can change results—this is normal in finance data replication.

## Formulas used (plain language)
- **Equity premium (level):** $\text{ERP}_t = R^{\text{SP}}_t - R^{\text{free}}_t$.
- **Lagged predictor:** $x_{t-1}$ (or $x_{t-2}$ for inflation).
- **OLS model:** $\text{ERP}_t = a + b\,x_{t-1} + \varepsilon_t$.
- **$R^2$ (in‑sample):** $R^2 = 1 - \frac{\sum \varepsilon_t^2}{\sum (y_t-\bar y)^2}$.
- **Adjusted $R^2$ (small‑sample correction):** $R^2_{\text{adj}} = R^2 - (1-R^2)\frac{T-k}{T-1}$, with $T$ observations and $k$ parameters.
- **OOS $R^2$:** compare the model’s MSE to the historical‑mean forecast MSE: $R^2_{\text{OOS}} = 1 - \frac{\text{MSE}_{\text{model}}}{\text{MSE}_{\text{mean}}}$.
- **OOS adjusted:** $R^2_{\text{OOS, adj}} = R^2_{\text{OOS}} - (1-R^2_{\text{OOS}})\frac{T-k}{T-1}$.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant

In [ ]:
def compute_monthly_stats(ts_df, indep, start, end, est_periods_OOS=240):
    """Compute the Table 3 metrics for one predictor (monthly data)."""
    dep = 'equity_premium'
    start_ts = pd.to_datetime(start)
    end_ts = pd.to_datetime(end)
    ts = ts_df.loc[start_ts:end_ts].copy()

    def lagged_x(df, var):
        # Most predictors are lagged 1 month; inflation uses 2 months due to release timing
        return df[var].shift(2) if var == 'infl' else df[var].shift(1)

    def is_r2_head(df, var, dep_var):
        # Simple OLS with lagged predictor and adjusted R^2
        x = lagged_x(df, var)
        idx = x.dropna().index
        y = df.loc[idx, dep_var]
        X = add_constant(x.loc[idx])
        reg = OLS(y, X).fit()
        T = len(y)
        k = int(reg.df_resid) + 1
        r2 = reg.rsquared
        r2_head = r2 - (1 - r2) * (T - k) / (T - 1)
        return r2_head, reg

    # In-sample model for equity premium
    IS_R2_head, reg = is_r2_head(ts, indep, dep)
    EXPECTED_SIGN = np.where(float(reg.params[indep]) > 0, 1, -1)
    # Expected sign is used to enforce sign constraints out-of-sample

    # OOS analysis (rolling, 20 years = 240 months)
    idx_all = ts_df.index
    start_pos = idx_all.searchsorted(start_ts, side='left')
    end_pos = idx_all.searchsorted(end_ts, side='right') - 1

    OOS_error_M, OOS_error_C = [], []

    for pos in range(start_pos + est_periods_OOS, end_pos):
        # Historical mean benchmark (no predictors)
        actual = ts_df.iloc[pos + 1][dep]
        avg_i = ts_df.iloc[start_pos:pos + 1][dep].mean()
        OOS_error_M.append(actual - avg_i)

        # Rolling regression using only past data up to 'pos'
        ts_train = ts_df.iloc[start_pos:pos + 1]
        x_train = lagged_x(ts_train, indep)
        idx_train = x_train.dropna().index
        y_train = ts_train.loc[idx_train, dep]
        X_train = add_constant(x_train.loc[idx_train])
        reg_oos = OLS(y_train, X_train).fit()

        # One-step-ahead prediction
        x_new = ts_df.iloc[pos][indep]
        x_new_const = add_constant(pd.DataFrame({indep: [x_new]}), has_constant='add')
        pred = reg_oos.predict(x_new_const)[0]
        OOS_error_C.append(pred - actual)

        # Sign restriction (if slope is wrong sign, set to zero)
        if (reg_oos.params[indep] * EXPECTED_SIGN) < 0:
            reg_oos.params[indep] = 0.0

    OOS_error_M = np.array(OOS_error_M)
    MSE_M = np.mean(OOS_error_M ** 2)
    OOS_R2_C = 1 - (np.mean(np.array(OOS_error_C) ** 2) / MSE_M)
    T = len(idx_train)
    k = T - 1
    OOS_R2_head = OOS_R2_C - (1 - OOS_R2_C) * (T - k) / (T - 1)

    return {
        'IS_R2_head': round(float(IS_R2_head) * 100, 2),
        'OOS_R2_head': round(float(OOS_R2_head) * 100, 2)
    }

In [ ]:
# Load monthly data
notebook_dir = Path.cwd()
file_path_monthly = notebook_dir / "data" / "GW05_original_monthly.csv"
data_monthly = pd.read_csv(file_path_monthly, sep=';', decimal=',')

# Parse date and keep the sample used in the original notebook
data_monthly['date'] = pd.to_datetime(data_monthly['yyyymm'], format='%Y%m')
data_monthly = data_monthly[data_monthly['date'] >= '1927-11-01'].copy()

# Construct predictor variables
data_monthly['dp'] = np.log(data_monthly['D12']) - np.log(data_monthly['Index'])
data_monthly['dy'] = np.log(data_monthly['D12']) - np.log(data_monthly['Index'].shift(1))
data_monthly['ep'] = np.log(data_monthly['E12']) - np.log(data_monthly['Index'])
data_monthly['de'] = np.log(data_monthly['D12']) - np.log(data_monthly['E12'])
data_monthly['e10p'] = np.log(data_monthly['E12'].rolling(window=120).mean()) - np.log(data_monthly['Index'])
data_monthly['tms'] = data_monthly['AAA'] - data_monthly['tbl']
data_monthly['dfy'] = data_monthly['BAA'] - data_monthly['AAA']

# Equity premium series (log and level)
data_monthly['log_equity_premium'] = np.log1p(data_monthly['CRSP_SPvw']) - np.log1p(data_monthly['Rfree'])
data_monthly['equity_premium'] = data_monthly['CRSP_SPvw'] - data_monthly['Rfree']
data_monthly['equity_premium_lag'] = data_monthly['equity_premium'].shift(1)
data_monthly['abs_ep_lag'] = data_monthly['equity_premium'].abs().shift(1)
data_monthly['sign_lag'] = (data_monthly['equity_premium'] > 0).astype(int).shift(1)
data_monthly = data_monthly.iloc[1:].copy()

# Set date index for helper functions
data_monthly.set_index('date', inplace=True)

In [ ]:
# Compute Table 3 statistics (two columns)
vars_list = ['de', 'svar', 'lty', 'ltr', 'infl', 'tms', 'tbl', 'dfy', 'dp', 'dy', 'ep', 'b/m', 'e10p', 'csp', 'ntis']
results = {}

for var in vars_list:
    if var == 'csp':
        stats = compute_monthly_stats(ts_df=data_monthly, indep=var, start='1937-05-01', end='2002-12-01', est_periods_OOS=240)
    else:
        stats = compute_monthly_stats(ts_df=data_monthly, indep=var, start='1927-12-01', end='2005-12-01', est_periods_OOS=240)
    results[var] = stats

df_results = pd.DataFrame.from_dict(results, orient='index')
df_results.index.name = 'variable'
df_results